# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
meta = dataset.metadata  # this is a Metadata object
print(f"{meta.name}: {meta.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all RecordSets available in the dataset

print("Available RecordSets:")
record_sets = list(dataset.record_sets)
for rs in record_sets:
    print(f"- @id: {rs.id}, name: {rs.name}")

# For each RecordSet, print its fields and columns by @id
for rs in record_sets:
    print(f"\nRecordSet: {rs.name}\n  @id: {rs.id}\n  Description: {rs.description}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - @id: {field.id}, name: {field.name}, dataType: {field.data_type}")
        cols = getattr(field, 'columns', [])
        for col in cols:
            # columns can be None if not mapped
            print(f"      Column @id: {col.id}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# For this analysis, select all the available RecordSet @id values
record_sets_ids = [rs.id for rs in record_sets]
dataframes = {}

for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
    else:
        print(f"No records found for RecordSet {record_set_id}")

if dataframes:
    # Pick the first populated RecordSet as default for downstream exploration
    main_record_set_id = next(iter(dataframes))
    print(f"DataFrame columns for {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No RecordSets loaded into DataFrames.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
import numpy as np
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# For demonstration, we'll pick a common numeric field for mental health: PHQ-9 total score (if present)
if dataframes:
    df = dataframes[main_record_set_id]
    # Try to locate a numeric mental health score field by column name -- e.g., 'phq9_total_score', 'PHQ9', etc.
    # You may need to inspect columns above to refine these names
    candidate_cols = [col for col in df.columns if 'phq' in col.lower() and 'score' in col.lower()]
    if candidate_cols:
        numeric_field = candidate_cols[0]
    else:
        # Fall back to any integer or float column
        numeric_types = [np.int64, np.float64]
        numeric_field = df.select_dtypes(include=numeric_types).columns[0] if len(df.select_dtypes(include=numeric_types).columns) > 0 else df.columns[0]

    print(f"Analyzing numeric field: {numeric_field}")
    # Filtering (e.g., PHQ-9 score > 10)
    threshold = 10
    if np.issubdtype(df[numeric_field].dtype, np.number):
        filtered_df = df[df[numeric_field] > threshold].copy()
    else:
        # Ensure numeric, coerce errors
        filtered_df = df[pd.to_numeric(df[numeric_field], errors='coerce') > threshold].copy()

    print(f"Filtered records with {numeric_field} > {threshold}: {len(filtered_df)}")
    display(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field}_normalized"] = (
        pd.to_numeric(filtered_df[numeric_field], errors='coerce') - pd.to_numeric(filtered_df[numeric_field], errors='coerce').mean()
    ) / pd.to_numeric(filtered_df[numeric_field], errors='coerce').std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Attempt to pick a group field -- e.g., 'gender', 'age_group', 'village', etc.
    group_candidates = [col for col in df.columns if any(x in col.lower() for x in ['gender', 'village', 'age group', 'education'])]
    if group_candidates:
        group_field = group_candidates[0]
        print(f"Grouping by {group_field}...")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(grouped_df)
    else:
        print("No suitable grouping field found in columns.")
else:
    print("No dataframe available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
# For interactive display in Jupyter
%matplotlib inline

if dataframes:
    df = dataframes[main_record_set_id]
    # Histogram of the selected numeric field
    plt.figure(figsize=(8, 4))
    sns.histplot(pd.to_numeric(df[numeric_field], errors='coerce').dropna(), kde=True, bins=15, color='skyblue')
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    # If group_field is selected, plot comparisons
    if 'group_field' in locals() and group_field in df.columns:
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f'{numeric_field} by {group_field}')
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset provides rich survey data on mental health indicators from Kilifi County, with fields including demographic and psychological assessment scores.
- Using `mlcroissant`, we can programmatically extract both metadata and records, and refer to entities using their stable `@id` identifiers.
- Example EDA and visualizations help to uncover groups with higher or lower mental health scores, identify potential outliers, and begin deeper analysis.

_For further study, consult the Croissant schema for precise field definitions, and consider cleaning or joining additional metadata as needed._